In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, recall_score, roc_auc_score, roc_curve
import gradio as gr
import re
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from matplotlib.patches import Patch

# 抑制sklearn警告
warnings.filterwarnings('ignore', category=UserWarning)
plt.style.use('default')
sns.set_palette("husl")

# ==================== 配置 ====================
class Config:
    SEED = 42
    DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    MODEL_PATH = './data/f1_zsl_model_final.pth'
    DATA_DIR = './data/'
    
    CAT_COLS = ['Grand Prix', 'Team', 'Driver', 'Nationality']
    NUM_COLS = ['year', 'Prev_Year_Driver_PTS', 'Prev_Year_Driver_Pos', 'Driver_Experience_Years',
                'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years',
                'Driver_Prev_Season_FL_Count', 'Is_Home_Race', 'Recent_3_Races_Avg_Pos', 
                'Performance_Trend', 'Consistency_Score']
    TARGET_COL = 'is_winner'
    
    # Zero-Shot Learning 特定配置
    EMB_DIM = 64  # 增加嵌入維度以支持更好的語義表示
    SEMANTIC_DIM = 128  # 語義特徵維度
    HIDDEN_DIM = 256  # 增加隱藏層維度
    DROPOUT_RATE = 0.4
    BATCH_SIZE = 256
    LEARNING_RATE = 5e-4
    N_EPOCHS = 60  # 增加訓練輪數
    PATIENCE = 15
    
    # ZSL相關超參數
    MARGIN = 0.5  # 用於triplet loss
    TEMPERATURE = 0.1  # 用於softmax溫度縮放

def set_seeds():
    """設定隨機種子"""
    np.random.seed(Config.SEED)
    torch.manual_seed(Config.SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(Config.SEED)

# ==================== 數據處理 ====================
def clean_string(text):
    """清理文字"""
    return re.sub(r'\s+', ' ', text).strip() if isinstance(text, str) else text

def get_country_from_gp(gp_name):
    """從GP名稱獲取國家代碼"""
    gp_map = {
        'British': 'GBR', 'Great Britain': 'GBR', 'Monaco': 'MON', 'Italian': 'ITA', 'Italy': 'ITA',
        'German': 'GER', 'Germany': 'GER', 'Belgian': 'BEL', 'Belgium': 'BEL', 'French': 'FRA', 'France': 'FRA',
        'Dutch': 'NED', 'Spanish': 'ESP', 'Spain': 'ESP', 'Brazilian': 'BRA', 'Brazil': 'BRA',
        'Japanese': 'JPN', 'Japan': 'JPN', 'Canadian': 'CAN', 'Canada': 'CAN', 'Austrian': 'AUT', 'Austria': 'AUT',
        'Hungarian': 'HUN', 'Hungary': 'HUN', 'Mexican': 'MEX', 'Mexico': 'MEX', 'Australian': 'AUS', 'Australia': 'AUS',
        'United States': 'USA', 'USA': 'USA', 'Swiss': 'SUI', 'Switzerland': 'SUI',
    }
    if not isinstance(gp_name, str):
        return None
    for key, country in gp_map.items():
        if key in gp_name:
            return country
    return None

def load_data():
    """載入所有CSV數據"""
    d = Config.DATA_DIR
    
    # 讀取數據
    winners = pd.read_csv(d + 'winners.csv', encoding='utf-8')
    drivers = pd.read_csv(d + 'drivers_updated.csv', encoding='utf-8')
    teams = pd.read_csv(d + 'teams_updated.csv', encoding='utf-8')
    fastest_laps = pd.read_csv(d + 'fastest_laps_updated.csv', encoding='utf-8')
    
    # 清理文字欄位
    for df in [winners, drivers, teams, fastest_laps]:
        for col in df.select_dtypes(include='object'):
            df[col] = df[col].map(clean_string)
    
    # 處理年份
    winners['year'] = pd.to_datetime(winners['Date'], errors='coerce').dt.year.astype(int)
    drivers.rename(columns={'Car': 'Team'}, inplace=True)
    
    for df in [drivers, teams, fastest_laps]:
        df['year'] = pd.to_numeric(df['year'], errors='coerce').astype(int)
        if 'Pos' in df.columns:
            df['Pos'] = pd.to_numeric(df['Pos'], errors='coerce')
    
    # 排除Indianapolis 500
    winners = winners[~winners['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    fastest_laps = fastest_laps[~fastest_laps['Grand Prix'].str.contains("Indianapolis 500", na=False)]
    
    return winners, drivers, teams, fastest_laps

def create_features(drivers, teams, fastest_laps):
    """創建所有特徵"""
    def_pos_drv, def_pos_team = 50, 20
    
    # 車手lag特徵
    drivers = drivers.sort_values(['Driver', 'year'])
    drivers['Prev_Year_Driver_PTS'] = drivers.groupby('Driver')['PTS'].shift(1).fillna(0)
    drivers['Prev_Year_Driver_Pos'] = drivers.groupby('Driver')['Pos'].shift(1).fillna(def_pos_drv)
    drivers['Driver_Experience_Years'] = drivers['year'] - drivers.groupby('Driver')['year'].transform('min')
    
    # 車隊lag特徵
    teams = teams.sort_values(['Team', 'year'])
    teams['Prev_Year_Team_PTS'] = teams.groupby('Team')['PTS'].shift(1).fillna(0)
    teams['Prev_Year_Team_Pos'] = teams.groupby('Team')['Pos'].shift(1).fillna(def_pos_team)
    teams['Team_Experience_Years'] = teams['year'] - teams.groupby('Team')['year'].transform('min')
    
    # 合併車隊特徵
    drivers = drivers.merge(teams[['Team', 'year', 'Prev_Year_Team_PTS', 'Prev_Year_Team_Pos', 'Team_Experience_Years']], 
                           on=['Team', 'year'], how='left').fillna(0)
    
    # 最快圈速特徵
    fl = fastest_laps.groupby(['year', 'Driver']).size().reset_index(name='FL_Count')
    fl['Driver_Prev_Season_FL_Count'] = fl.groupby('Driver')['FL_Count'].shift(1).fillna(0)
    drivers = drivers.merge(fl[['Driver', 'year', 'Driver_Prev_Season_FL_Count']], on=['Driver', 'year'], how='left').fillna(0)
    
    # Momentum特徵
    drivers['Recent_3_Races_Avg_Pos'] = drivers.groupby('Driver')['Pos'].rolling(window=3, min_periods=1).mean().reset_index(0, drop=True).fillna(def_pos_drv)
    
    # 安全計算表現趨勢
    drivers['PTS_prev'] = drivers.groupby('Driver')['PTS'].shift(1)
    drivers['Performance_Trend'] = 0.0
    mask = (drivers['PTS_prev'].notna()) & (drivers['PTS_prev'] > 0)
    drivers.loc[mask, 'Performance_Trend'] = ((drivers.loc[mask, 'PTS'] - drivers.loc[mask, 'PTS_prev']) / drivers.loc[mask, 'PTS_prev']).clip(-2.0, 2.0)
    drivers.drop('PTS_prev', axis=1, inplace=True)
    
    # 一致性評分
    drivers['Pos_Rolling_Std'] = drivers.groupby('Driver')['Pos'].rolling(window=3, min_periods=1).std().reset_index(0, drop=True).fillna(0.0)
    drivers['Consistency_Score'] = 1.0 / (1.0 + drivers['Pos_Rolling_Std'])
    
    # 確保數值安全
    for col in ['Recent_3_Races_Avg_Pos', 'Performance_Trend', 'Consistency_Score']:
        drivers[col] = drivers[col].replace([float('inf'), float('-inf')], 0.0).fillna(0.0)
    
    return drivers

def build_dataset(winners, drivers):
    """構建建模數據集"""
    data = []
    drivers_by_year = {y: g for y, g in drivers.groupby('year')}
    
    for _, race in winners.iterrows():
        year, gp, winner = race['year'], race['Grand Prix'], race['Winner']
        if year not in drivers_by_year:
            continue
            
        race_country = get_country_from_gp(gp)
        for _, driver in drivers_by_year[year].iterrows():
            is_home = 1 if race_country and driver['Nationality'] == race_country else 0
            
            data.append({
                'year': year, 'Grand Prix': gp, 'Driver': driver['Driver'], 
                'Team': driver['Team'], 'Nationality': driver['Nationality'],
                'Prev_Year_Driver_PTS': driver['Prev_Year_Driver_PTS'],
                'Prev_Year_Driver_Pos': driver['Prev_Year_Driver_Pos'],
                'Driver_Experience_Years': driver['Driver_Experience_Years'],
                'Prev_Year_Team_PTS': driver['Prev_Year_Team_PTS'],
                'Prev_Year_Team_Pos': driver['Prev_Year_Team_Pos'],
                'Team_Experience_Years': driver['Team_Experience_Years'],
                'Driver_Prev_Season_FL_Count': driver['Driver_Prev_Season_FL_Count'],
                'Recent_3_Races_Avg_Pos': driver['Recent_3_Races_Avg_Pos'],
                'Performance_Trend': driver['Performance_Trend'],
                'Consistency_Score': driver['Consistency_Score'],
                'Is_Home_Race': is_home,
                'is_winner': int(driver['Driver'] == winner)
            })
    
    return pd.DataFrame(data)

def create_semantic_attributes():
    """創建語義屬性 - ZSL的核心組件"""
    # 定義獲勝者的語義屬性（可以根據F1領域知識擴展）
    winner_attributes = {
        'high_performance': 1.0,      # 高性能表現
        'experienced': 1.0,           # 經驗豐富
        'consistent': 1.0,            # 穩定性
        'competitive_car': 1.0,       # 有競爭力的賽車
        'home_advantage': 0.5,        # 主場優勢（部分情況）
        'momentum': 1.0,              # 近期表現趨勢
        'fastest_lap_ability': 0.8,   # 最快圈速能力
        'championship_contender': 1.0  # 冠軍爭奪者
    }
    
    non_winner_attributes = {
        'high_performance': 0.2,
        'experienced': 0.4,
        'consistent': 0.3,
        'competitive_car': 0.3,
        'home_advantage': 0.1,
        'momentum': 0.2,
        'fastest_lap_ability': 0.3,
        'championship_contender': 0.1
    }
    
    return winner_attributes, non_winner_attributes

def map_features_to_semantics(sample_features):
    """將樣本特徵映射到語義屬性空間"""
    # 這個函數將數值特徵轉換為語義屬性
    semantic_vector = {}
    
    # 高性能表現 (基於積分和排名)
    prev_pts = sample_features.get('Prev_Year_Driver_PTS', 0)
    prev_pos = sample_features.get('Prev_Year_Driver_Pos', 50)
    semantic_vector['high_performance'] = min(1.0, prev_pts / 300.0) * (1.0 - min(1.0, prev_pos / 20.0))
    
    # 經驗豐富
    exp_years = sample_features.get('Driver_Experience_Years', 0)
    semantic_vector['experienced'] = min(1.0, exp_years / 15.0)
    
    # 穩定性
    consistency = sample_features.get('Consistency_Score', 0)
    semantic_vector['consistent'] = consistency
    
    # 有競爭力的賽車
    team_pts = sample_features.get('Prev_Year_Team_PTS', 0)
    team_pos = sample_features.get('Prev_Year_Team_Pos', 20)
    semantic_vector['competitive_car'] = min(1.0, team_pts / 500.0) * (1.0 - min(1.0, team_pos / 10.0))
    
    # 主場優勢
    semantic_vector['home_advantage'] = sample_features.get('Is_Home_Race', 0)
    
    # 近期表現趨勢
    trend = sample_features.get('Performance_Trend', 0)
    avg_pos = sample_features.get('Recent_3_Races_Avg_Pos', 50)
    semantic_vector['momentum'] = max(0.0, trend) * (1.0 - min(1.0, avg_pos / 20.0))
    
    # 最快圈速能力
    fl_count = sample_features.get('Driver_Prev_Season_FL_Count', 0)
    semantic_vector['fastest_lap_ability'] = min(1.0, fl_count / 5.0)
    
    # 冠軍爭奪者 (綜合指標)
    semantic_vector['championship_contender'] = (
        semantic_vector['high_performance'] * 0.4 +
        semantic_vector['competitive_car'] * 0.3 +
        semantic_vector['experienced'] * 0.2 +
        semantic_vector['consistent'] * 0.1
    )
    
    return semantic_vector

def preprocess_data(train_df, test_df):
    """預處理數據 - 包含語義屬性生成"""
    # 填補缺失值
    for col in Config.CAT_COLS:
        train_df[col] = train_df[col].fillna('Unknown')
        test_df[col] = test_df[col].fillna('Unknown')
    for col in Config.NUM_COLS:
        train_df[col] = train_df[col].fillna(0)
        test_df[col] = test_df[col].fillna(0)
    
    # 編碼分類特徵
    encoders, cat_dims = {}, {}
    for col in Config.CAT_COLS:
        le = LabelEncoder()
        train_df[col] = le.fit_transform(train_df[col].astype(str))
        test_df[col] = test_df[col].map(lambda x: le.transform([str(x)])[0] if str(x) in le.classes_ else len(le.classes_))
        encoders[col] = le
        cat_dims[col] = len(le.classes_) + 1
    
    # 標準化數值特徵
    scaler = StandardScaler()
    train_df[Config.NUM_COLS] = scaler.fit_transform(train_df[Config.NUM_COLS].values)
    test_df[Config.NUM_COLS] = scaler.transform(test_df[Config.NUM_COLS].values)
    
    # 創建語義屬性
    winner_attrs, non_winner_attrs = create_semantic_attributes()
    
    return train_df, test_df, encoders, scaler, cat_dims, winner_attrs, non_winner_attrs

# ==================== Zero-Shot Learning 模型 ====================
class SemanticEmbedding(nn.Module):
    """語義嵌入層 - 將特徵映射到語義空間"""
    def __init__(self, input_dim, semantic_dim):
        super().__init__()
        self.projection = nn.Sequential(
            nn.Linear(input_dim, semantic_dim * 2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(semantic_dim * 2, semantic_dim),
            nn.Tanh()  # 限制輸出範圍
        )
    
    def forward(self, x):
        return self.projection(x)

class ZSLLoss(nn.Module):
    """Zero-Shot Learning 損失函數 - 結合分類損失和語義對齊損失"""
    def __init__(self, margin=0.5, semantic_weight=0.3):
        super().__init__()
        self.margin = margin
        self.semantic_weight = semantic_weight
        self.bce_loss = nn.BCEWithLogitsLoss()
        self.mse_loss = nn.MSELoss()
    
    def forward(self, pred_logits, semantic_pred, target_class, target_semantic):
        # 分類損失
        classification_loss = self.bce_loss(pred_logits, target_class.unsqueeze(1))
        
        # 語義對齊損失
        semantic_loss = self.mse_loss(semantic_pred, target_semantic)
        
        # 總損失
        total_loss = classification_loss + self.semantic_weight * semantic_loss
        
        return total_loss, classification_loss, semantic_loss

class F1Dataset(Dataset):
    def __init__(self, df, winner_attrs, non_winner_attrs):
        self.x_cat = df[Config.CAT_COLS].values
        self.x_num = df[Config.NUM_COLS].values
        self.y = df[Config.TARGET_COL].values
        
        # 生成語義屬性標籤
        self.semantic_targets = []
        for idx in range(len(df)):
            row_dict = {col: df.iloc[idx][col] for col in Config.NUM_COLS}
            
            if self.y[idx] == 1:  # 獲勝者
                target_attrs = list(winner_attrs.values())
            else:  # 非獲勝者
                target_attrs = list(non_winner_attrs.values())
            
            self.semantic_targets.append(target_attrs)
        
        self.semantic_targets = np.array(self.semantic_targets)
    
    def __len__(self):
        return len(self.y)
    
    def __getitem__(self, idx):
        return (torch.tensor(self.x_cat[idx], dtype=torch.long),
                torch.tensor(self.x_num[idx], dtype=torch.float32),
                torch.tensor(self.y[idx], dtype=torch.float32),
                torch.tensor(self.semantic_targets[idx], dtype=torch.float32))

class F1ZSLModel(nn.Module):
    """F1 Zero-Shot Learning 模型"""
    def __init__(self, cat_dims, num_feats, semantic_attrs_count=8):
        super().__init__()
        
        # 嵌入層
        self.embeddings = nn.ModuleList([nn.Embedding(dim, Config.EMB_DIM) for dim in cat_dims])
        self.bn_num = nn.BatchNorm1d(num_feats)
        
        # 特徵提取器
        total_input_dim = len(cat_dims) * Config.EMB_DIM + num_feats
        self.feature_extractor = nn.Sequential(
            nn.Linear(total_input_dim, Config.HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(Config.HIDDEN_DIM, Config.HIDDEN_DIM),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE)
        )
        
        # 語義嵌入
        self.semantic_embedding = SemanticEmbedding(Config.HIDDEN_DIM, Config.SEMANTIC_DIM)
        
        # 語義屬性預測器
        self.semantic_predictor = nn.Sequential(
            nn.Linear(Config.SEMANTIC_DIM, Config.SEMANTIC_DIM // 2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(Config.SEMANTIC_DIM // 2, semantic_attrs_count),
            nn.Sigmoid()  # 語義屬性值在[0,1]範圍
        )
        
        # 分類器 (基於語義特徵)
        self.classifier = nn.Sequential(
            nn.Linear(Config.SEMANTIC_DIM + semantic_attrs_count, Config.HIDDEN_DIM // 2),
            nn.ReLU(),
            nn.Dropout(Config.DROPOUT_RATE),
            nn.Linear(Config.HIDDEN_DIM // 2, 1)
        )
        
        # 語義原型 (獲勝者和非獲勝者的語義表示)
        self.winner_prototype = nn.Parameter(torch.randn(semantic_attrs_count))
        self.non_winner_prototype = nn.Parameter(torch.randn(semantic_attrs_count))
    
    def forward(self, x_cat, x_num):
        # 嵌入分類特徵
        cat_emb = torch.cat([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], 1)
        
        # 正規化數值特徵
        num_norm = self.bn_num(x_num)
        
        # 合併特徵
        combined = torch.cat([cat_emb, num_norm], 1)
        
        # 特徵提取
        features = self.feature_extractor(combined)
        
        # 語義嵌入
        semantic_features = self.semantic_embedding(features)
        
        # 語義屬性預測
        semantic_pred = self.semantic_predictor(semantic_features)
        
        # 結合語義特徵和屬性進行分類
        combined_semantic = torch.cat([semantic_features, semantic_pred], 1)
        classification_logits = self.classifier(combined_semantic)
        
        return classification_logits, semantic_pred, semantic_features

    def zero_shot_predict(self, x_cat, x_num, winner_attrs, non_winner_attrs):
        """零樣本預測 - 使用語義原型進行預測"""
        self.eval()
        with torch.no_grad():
            _, semantic_pred, semantic_features = self.forward(x_cat, x_num)
            
            # 計算與獲勝者和非獲勝者原型的相似度
            winner_proto = torch.tensor(list(winner_attrs.values()), device=x_cat.device).float()
            non_winner_proto = torch.tensor(list(non_winner_attrs.values()), device=x_cat.device).float()
            
            # 餘弦相似度
            winner_sim = F.cosine_similarity(semantic_pred, winner_proto.unsqueeze(0), dim=1)
            non_winner_sim = F.cosine_similarity(semantic_pred, non_winner_proto.unsqueeze(0), dim=1)
            
            # 軟最大值決策
            similarities = torch.stack([non_winner_sim, winner_sim], dim=1)
            probabilities = F.softmax(similarities / Config.TEMPERATURE, dim=1)
            
            return probabilities[:, 1]  # 返回獲勝概率

def train_model(model, train_loader, test_loader, winner_attrs, non_winner_attrs):
    """訓練零樣本學習模型"""
    optimizer = torch.optim.AdamW(model.parameters(), lr=Config.LEARNING_RATE, weight_decay=1e-4)
    criterion = ZSLLoss(margin=Config.MARGIN)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5)
    
    best_loss = float('inf')
    no_improve = 0
    train_losses, test_losses = [], []
    
    for epoch in range(Config.N_EPOCHS):
        # 訓練
        model.train()
        train_loss = 0
        train_cls_loss = 0
        train_sem_loss = 0
        
        for x_cat, x_num, y, semantic_target in train_loader:
            x_cat = x_cat.to(Config.DEVICE)
            x_num = x_num.to(Config.DEVICE) 
            y = y.to(Config.DEVICE)
            semantic_target = semantic_target.to(Config.DEVICE)
            
            optimizer.zero_grad()
            
            classification_logits, semantic_pred, _ = model(x_cat, x_num)
            total_loss, cls_loss, sem_loss = criterion(
                classification_logits, semantic_pred, y, semantic_target
            )
            
            total_loss.backward()
            optimizer.step()
            
            train_loss += total_loss.item()
            train_cls_loss += cls_loss.item()
            train_sem_loss += sem_loss.item()
        
        # 驗證
        model.eval()
        test_loss = 0
        test_cls_loss = 0
        test_sem_loss = 0
        
        with torch.no_grad():
            for x_cat, x_num, y, semantic_target in test_loader:
                x_cat = x_cat.to(Config.DEVICE)
                x_num = x_num.to(Config.DEVICE)
                y = y.to(Config.DEVICE)
                semantic_target = semantic_target.to(Config.DEVICE)
                
                classification_logits, semantic_pred, _ = model(x_cat, x_num)
                total_loss, cls_loss, sem_loss = criterion(
                    classification_logits, semantic_pred, y, semantic_target
                )
                
                test_loss += total_loss.item()
                test_cls_loss += cls_loss.item()
                test_sem_loss += sem_loss.item()
        
        train_loss /= len(train_loader)
        test_loss /= len(test_loader)
        train_cls_loss /= len(train_loader)
        test_cls_loss /= len(test_loader)
        train_sem_loss /= len(train_loader)
        test_sem_loss /= len(test_loader)
        
        train_losses.append(train_loss)
        test_losses.append(test_loss)
        
        print(f"Epoch {epoch+1} | Train: {train_loss:.4f} (Cls: {train_cls_loss:.4f}, Sem: {train_sem_loss:.4f}) | "
              f"Test: {test_loss:.4f} (Cls: {test_cls_loss:.4f}, Sem: {test_sem_loss:.4f})")
        
        scheduler.step(test_loss)
        
        # 早停
        if test_loss < best_loss:
            best_loss = test_loss
            torch.save({
                'model_state_dict': model.state_dict(),
                'winner_attrs': winner_attrs,
                'non_winner_attrs': non_winner_attrs
            }, Config.MODEL_PATH)
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= Config.PATIENCE:
                print("Early stopping")
                break
    
    # 保存訓練曲線
    plt.figure(figsize=(10, 6))
    plt.plot(train_losses, label='Train Total Loss', linewidth=2)
    plt.plot(test_losses, label='Test Total Loss', linewidth=2)
    plt.xlabel('Epoch', fontsize=12)
    plt.ylabel('Loss', fontsize=12)
    plt.title('Zero-Shot Learning Training and Validation Loss', fontsize=16, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('./data/zsl_loss_curve_final.png', dpi=300, bbox_inches='tight')
    plt.close()

# ==================== 評估和繪圖功能 ====================
def calculate_feature_importance(model, test_loader, feature_names):
    """計算特徵重要性（使用梯度方法）"""
    model.eval()
    importances = []
    
    for x_cat, x_num, y, _ in test_loader:
        x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
        x_num.requires_grad_(True)
        
        output, _, _ = model(x_cat, x_num)
        loss = F.binary_cross_entropy_with_logits(output, y.unsqueeze(1))
        
        grad = torch.autograd.grad(loss, x_num, create_graph=False)[0]
        importances.append(grad.abs().mean(dim=0).cpu().detach().numpy())
    
    avg_importance = np.mean(importances, axis=0)
    return dict(zip(feature_names, avg_importance))

def calculate_top3_accuracy_by_race(model, test_df, winner_attrs, non_winner_attrs):
    """計算每場比賽的Top-3準確率（零樣本版本）"""
    model.eval()
    race_groups = test_df.groupby(['year', 'Grand Prix'])
    top3_correct = 0
    total_races = 0
    
    for (year, gp), race_data in race_groups:
        # 獲取該場比賽的所有預測
        x_cat = torch.tensor(race_data[Config.CAT_COLS].values, dtype=torch.long).to(Config.DEVICE)
        x_num = torch.tensor(race_data[Config.NUM_COLS].values, dtype=torch.float32).to(Config.DEVICE)
        
        with torch.no_grad():
            # 使用零樣本預測
            probs = model.zero_shot_predict(x_cat, x_num, winner_attrs, non_winner_attrs).cpu().numpy()
        
        # 找出真實獲勝者
        true_winner_idx = race_data[race_data[Config.TARGET_COL] == 1].index
        if len(true_winner_idx) == 0:
            continue
        
        # 找出預測概率最高的前3名
        race_indices = race_data.index.tolist()
        prob_with_idx = list(zip(probs, race_indices))
        prob_with_idx.sort(key=lambda x: x[0], reverse=True)
        top3_indices = [idx for _, idx in prob_with_idx[:3]]
        
        # 檢查真實獲勝者是否在前3名中
        if any(idx in top3_indices for idx in true_winner_idx):
            top3_correct += 1
        
        total_races += 1
    
    return top3_correct / total_races if total_races > 0 else 0

def plot_confusion_matrix(y_true, y_pred, save_path='./data/zsl_confusion_matrix.png'):
    """繪製混淆矩陣熱力圖"""
    cm = confusion_matrix(y_true, y_pred)
    
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=['Not Winner', 'Winner'],
                yticklabels=['Not Winner', 'Winner'])
    plt.title('Zero-Shot Learning Confusion Matrix', fontsize=16, fontweight='bold')
    plt.xlabel('Predicted Label', fontsize=12)
    plt.ylabel('True Label', fontsize=12)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_roc_curve(y_true, y_probs, save_path='./data/zsl_roc_curve.png'):
    """繪製ROC曲線"""
    fpr, tpr, thresholds = roc_curve(y_true, y_probs)
    auc_score = roc_auc_score(y_true, y_probs)
    
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='darkorange', lw=2, 
             label=f'ZSL ROC Curve (AUC = {auc_score:.3f})')
    plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--', 
             label='Random Classifier')
    plt.xlim([0.0, 1.0])
    plt.ylim([0.0, 1.05])
    plt.xlabel('False Positive Rate', fontsize=12)
    plt.ylabel('True Positive Rate', fontsize=12)
    plt.title('Zero-Shot Learning ROC Curve', fontsize=16, fontweight='bold')
    plt.legend(loc="lower right")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_feature_importance(importance_dict, save_path='./data/zsl_feature_importance.png'):
    """繪製特徵重要性圖"""
    features = list(importance_dict.keys())
    importances = list(importance_dict.values())
    
    # 排序
    sorted_idx = np.argsort(importances)
    features_sorted = [features[i] for i in sorted_idx]
    importances_sorted = [importances[i] for i in sorted_idx]
    
    plt.figure(figsize=(10, 8))
    colors = plt.cm.viridis(np.linspace(0, 1, len(features_sorted)))
    bars = plt.barh(range(len(features_sorted)), importances_sorted, color=colors)
    plt.yticks(range(len(features_sorted)), features_sorted)
    plt.xlabel('Feature Importance (ZSL Gradient-based)', fontsize=12)
    plt.title('Zero-Shot Learning Feature Importance Analysis', fontsize=16, fontweight='bold')
    plt.grid(True, alpha=0.3, axis='x')
    
    # 添加數值標籤
    for i, (bar, imp) in enumerate(zip(bars, importances_sorted)):
        plt.text(bar.get_width() + max(importances_sorted) * 0.01, 
                bar.get_y() + bar.get_height()/2, 
                f'{imp:.4f}', va='center', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_semantic_space_analysis(model, test_loader, winner_attrs, non_winner_attrs, save_path='./data/zsl_semantic_space.png'):
    """繪製語義空間分析圖"""
    model.eval()
    all_semantic_features = []
    all_labels = []
    
    with torch.no_grad():
        for x_cat, x_num, y, _ in test_loader:
            x_cat, x_num = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE)
            _, _, semantic_features = model(x_cat, x_num)
            
            all_semantic_features.append(semantic_features.cpu().numpy())
            all_labels.append(y.numpy())
    
    all_semantic_features = np.vstack(all_semantic_features)
    all_labels = np.concatenate(all_labels)
    
    # 使用PCA降維到2D進行視覺化
    from sklearn.decomposition import PCA
    pca = PCA(n_components=2)
    semantic_2d = pca.fit_transform(all_semantic_features)
    
    plt.figure(figsize=(12, 8))
    
    # 繪製樣本點
    winners_mask = all_labels == 1
    plt.scatter(semantic_2d[~winners_mask, 0], semantic_2d[~winners_mask, 1], 
                c='blue', alpha=0.6, s=20, label='Non-Winners')
    plt.scatter(semantic_2d[winners_mask, 0], semantic_2d[winners_mask, 1], 
                c='red', alpha=0.8, s=30, label='Winners')
    
    # 繪製語義原型
    winner_proto = np.array(list(winner_attrs.values())).reshape(1, -1)
    non_winner_proto = np.array(list(non_winner_attrs.values())).reshape(1, -1)
    
    # 由於原型在不同空間，我們用平均值來近似表示
    winner_center = semantic_2d[winners_mask].mean(axis=0) if winners_mask.sum() > 0 else np.array([0, 0])
    non_winner_center = semantic_2d[~winners_mask].mean(axis=0)
    
    plt.scatter(*winner_center, c='darkred', s=200, marker='*', 
                label='Winner Prototype', edgecolor='black', linewidth=2)
    plt.scatter(*non_winner_center, c='darkblue', s=200, marker='*', 
                label='Non-Winner Prototype', edgecolor='black', linewidth=2)
    
    plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.2%} variance)', fontsize=12)
    plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.2%} variance)', fontsize=12)
    plt.title('Zero-Shot Learning Semantic Space Visualization', fontsize=16, fontweight='bold')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_prediction_distribution(y_true, y_probs, save_path='./data/zsl_prediction_distribution.png'):
    """繪製預測概率分布圖"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 左圖：按真實標籤分別顯示預測概率分布
    winners_probs = [prob for prob, label in zip(y_probs, y_true) if label == 1]
    non_winners_probs = [prob for prob, label in zip(y_probs, y_true) if label == 0]
    
    ax1.hist(non_winners_probs, bins=50, alpha=0.7, label='Non-Winners', 
             color='skyblue', density=True)
    ax1.hist(winners_probs, bins=50, alpha=0.7, label='Winners', 
             color='salmon', density=True)
    ax1.set_xlabel('Predicted Probability', fontsize=12)
    ax1.set_ylabel('Density', fontsize=12)
    ax1.set_title('ZSL Prediction Distribution by True Label', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 右圖：預測概率的整體分布
    ax2.hist(y_probs, bins=50, alpha=0.7, color='lightgreen', edgecolor='black')
    ax2.axvline(x=0.5, color='red', linestyle='--', linewidth=2, label='Decision Threshold (0.5)')
    ax2.set_xlabel('Predicted Probability', fontsize=12)
    ax2.set_ylabel('Frequency', fontsize=12)
    ax2.set_title('ZSL Overall Prediction Distribution', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def plot_top_predictions_analysis(y_true, y_probs, save_path='./data/zsl_top_predictions_analysis.png'):
    """分析預測概率最高的樣本"""
    # 將概率和真實標籤配對並排序
    prob_label_pairs = list(zip(y_probs, y_true))
    prob_label_pairs.sort(key=lambda x: x[0], reverse=True)
    
    # 取前100個最高概率的預測
    top_n = min(100, len(prob_label_pairs))
    top_probs = [pair[0] for pair in prob_label_pairs[:top_n]]
    top_labels = [pair[1] for pair in prob_label_pairs[:top_n]]
    
    plt.figure(figsize=(12, 8))
    
    # 創建顏色映射
    colors = ['red' if label == 1 else 'blue' for label in top_labels]
    
    plt.scatter(range(top_n), top_probs, c=colors, alpha=0.6, s=50)
    plt.xlabel(f'Rank (Top {top_n} ZSL Predictions)', fontsize=12)
    plt.ylabel('Predicted Probability', fontsize=12)
    plt.title('Zero-Shot Learning Top Predictions Analysis', fontsize=16, fontweight='bold')
    
    # 添加圖例
    legend_elements = [Patch(facecolor='red', label='True Winners'),
                      Patch(facecolor='blue', label='True Non-Winners')]
    plt.legend(handles=legend_elements)
    
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()

def evaluate_model(model, test_loader, test_df, winner_attrs, non_winner_attrs):
    """評估零樣本學習模型"""
    if os.path.exists(Config.MODEL_PATH):
        checkpoint = torch.load(Config.MODEL_PATH, map_location=Config.DEVICE)
        model.load_state_dict(checkpoint['model_state_dict'])
        winner_attrs = checkpoint['winner_attrs']
        non_winner_attrs = checkpoint['non_winner_attrs']
    
    model.eval()
    y_true, y_pred, y_probs = [], [], []
    
    with torch.no_grad():
        for x_cat, x_num, y, _ in test_loader:
            x_cat, x_num, y = x_cat.to(Config.DEVICE), x_num.to(Config.DEVICE), y.to(Config.DEVICE)
            
            # 使用零樣本預測
            probs = model.zero_shot_predict(x_cat, x_num, winner_attrs, non_winner_attrs)
            pred = (probs > 0.5).int()
            
            y_true.extend(y.cpu().numpy())
            y_pred.extend(pred.cpu().numpy() if pred.ndim > 0 else [pred.item()])
            y_probs.extend(probs.cpu().numpy())
    
    # 基本準確率
    accuracy = accuracy_score(y_true, y_pred)
    print(f"Zero-Shot Binary Classification Accuracy: {accuracy:.4f}")
    
    # Recall計算
    recall = recall_score(y_true, y_pred, zero_division=0)
    print(f"Zero-Shot Recall (Winner Detection): {recall:.4f}")
    
    # AUC計算
    auc_score = roc_auc_score(y_true, y_probs)
    print(f"Zero-Shot AUC Score: {auc_score:.4f}")
    
    # Top-3準確率（按比賽分組計算）
    top3_accuracy = calculate_top3_accuracy_by_race(model, test_df, winner_attrs, non_winner_attrs)
    print(f"Zero-Shot Top-3 Accuracy (Race-wise): {top3_accuracy:.4f}")
    
    # 詳細分類報告
    print("\nZero-Shot Learning Classification Report:")
    print(classification_report(y_true, y_pred, target_names=['Not Winner', 'Winner'], zero_division=0))
    
    # 混淆矩陣
    print("\nZero-Shot Learning Confusion Matrix:")
    cm = confusion_matrix(y_true, y_pred)
    print(cm)
    
    # ==================== 生成所有圖表 ====================
    print("\nGenerating Zero-Shot Learning analysis plots...")
    
    # 1. 混淆矩陣熱力圖
    plot_confusion_matrix(y_true, y_pred)
    print("✓ ZSL Confusion matrix heatmap saved")
    
    # 2. ROC曲線
    plot_roc_curve(y_true, y_probs)
    print("✓ ZSL ROC curve saved")
    
    # 3. 特徵重要性
    try:
        importance_dict = calculate_feature_importance(model, test_loader, Config.NUM_COLS)
        plot_feature_importance(importance_dict)
        print("✓ ZSL Feature importance plot saved")
    except Exception as e:
        print(f"⚠ ZSL Feature importance calculation failed: {e}")
    
    # 4. 語義空間分析
    try:
        plot_semantic_space_analysis(model, test_loader, winner_attrs, non_winner_attrs)
        print("✓ ZSL Semantic space analysis saved")
    except Exception as e:
        print(f"⚠ ZSL Semantic space analysis failed: {e}")
    
    # 5. 預測概率分布
    plot_prediction_distribution(y_true, y_probs)
    print("✓ ZSL Prediction distribution plots saved")
    
    # 6. 頂部預測分析
    plot_top_predictions_analysis(y_true, y_probs)
    print("✓ ZSL Top predictions analysis saved")
    
    # 輸出摘要
    print("\n" + "="*60)
    print("ZERO-SHOT LEARNING EVALUATION SUMMARY")
    print("="*60)
    print(f"Binary Classification Accuracy: {accuracy:.4f}")
    print(f"Top-3 Accuracy (Race-wise): {top3_accuracy:.4f}")
    print(f"Recall (Winner Detection): {recall:.4f}")
    print(f"AUC Score: {auc_score:.4f}")
    print("="*60)
    print("\nAll Zero-Shot Learning analysis plots have been saved to ./data/ directory")

# ==================== 預測界面 ====================
class F1ZSLPredictor:
    def __init__(self):
        self.model = None
        self.encoders = {}
        self.scaler = None
        self.driver_data = {}
        self.years = []
        self.gps = []
        self.winner_attrs = None
        self.non_winner_attrs = None
    
    def setup(self, model, encoders, scaler, drivers, winners, winner_attrs, non_winner_attrs):
        """設置零樣本預測器"""
        self.model = model
        self.encoders = encoders
        self.scaler = scaler
        self.winner_attrs = winner_attrs
        self.non_winner_attrs = non_winner_attrs
        
        for year, group in drivers.groupby('year'):
            self.driver_data[year] = group.to_dict('records')
        
        self.years = sorted(drivers['year'].unique())
        self.gps = sorted(winners['Grand Prix'].unique())
    
    def predict(self, year_input, gp_input):
        """零樣本預測獲勝概率"""
        try:
            year = int(year_input)
        except:
            return "Error: Invalid year"
        
        if not gp_input or year not in self.driver_data:
            return "Error: No data available"
        
        self.model.eval()
        results = []
        
        for driver_info in self.driver_data[year]:
            # 分類特徵
            cat = []
            for col in Config.CAT_COLS:
                val = str(driver_info.get(col, 'Unknown'))
                if col == 'Grand Prix':
                    val = gp_input
                
                if val in self.encoders[col].classes_:
                    encoded = self.encoders[col].transform([val])[0]
                else:
                    encoded = len(self.encoders[col].classes_)
                cat.append(encoded)
            
            # 數值特徵
            num = []
            for col in Config.NUM_COLS:
                if col == 'Is_Home_Race':
                    race_country = get_country_from_gp(gp_input)
                    driver_nationality = driver_info.get('Nationality', '')
                    val = 1.0 if race_country and driver_nationality == race_country else 0.0
                else:
                    val = float(driver_info.get(col, 0))
                num.append(val)
            
            # 零樣本預測
            x_num = torch.tensor(self.scaler.transform(np.array([num])), dtype=torch.float32).to(Config.DEVICE)
            x_cat = torch.tensor([cat], dtype=torch.long).to(Config.DEVICE)
            
            with torch.no_grad():
                prob = self.model.zero_shot_predict(x_cat, x_num, self.winner_attrs, self.non_winner_attrs).cpu().item()
            
            results.append((driver_info.get('Driver', 'N/A'), driver_info.get('Team', 'N/A'), prob))
        
        # 排序並返回前5名
        results.sort(key=lambda x: x[2], reverse=True)
        output = f"Zero-Shot Learning Predictions for {gp_input}, {year}:\n"
        for i, (driver, team, prob) in enumerate(results[:5], 1):
            output += f"{i}. {driver} ({team}): {prob:.2%}\n"
        
        return output.strip()
    
    def create_interface(self):
        """創建零樣本學習Gradio界面"""
        with gr.Blocks(theme=gr.themes.Soft()) as demo:
            gr.Markdown("# F1 Zero-Shot Learning Winner Predictor")
            gr.Markdown("### 使用語義屬性進行零樣本學習的F1賽車獲勝預測")
            
            with gr.Row():
                year_dd = gr.Dropdown(label="Year", choices=self.years, value=self.years[-1] if self.years else None)
                gp_dd = gr.Dropdown(label="Grand Prix", choices=self.gps, value=self.gps[0] if self.gps else None)
            
            predict_btn = gr.Button("Zero-Shot Predict Winners", variant="primary")
            output_tb = gr.Textbox(label="Top 5 ZSL Predictions", lines=6, interactive=False)
            
            predict_btn.click(self.predict, inputs=[year_dd, gp_dd], outputs=[output_tb])
            
            with gr.Accordion("Zero-Shot Learning Training Analysis & Results", open=False):
                gr.Markdown("### Zero-Shot Learning 模型分析結果")
                
                with gr.Row():
                    with gr.Column():
                        if os.path.exists("./data/zsl_loss_curve_final.png"):
                            gr.Image(value="./data/zsl_loss_curve_final.png", label="ZSL Training Loss Curve")
                        if os.path.exists("./data/zsl_confusion_matrix.png"):
                            gr.Image(value="./data/zsl_confusion_matrix.png", label="ZSL Confusion Matrix")
                    
                    with gr.Column():
                        if os.path.exists("./data/zsl_roc_curve.png"):
                            gr.Image(value="./data/zsl_roc_curve.png", label="ZSL ROC Curve")
                        if os.path.exists("./data/zsl_feature_importance.png"):
                            gr.Image(value="./data/zsl_feature_importance.png", label="ZSL Feature Importance")
                
                with gr.Row():
                    if os.path.exists("./data/zsl_semantic_space.png"):
                        gr.Image(value="./data/zsl_semantic_space.png", label="ZSL Semantic Space Analysis")
                
                with gr.Row():
                    if os.path.exists("./data/zsl_prediction_distribution.png"):
                        gr.Image(value="./data/zsl_prediction_distribution.png", label="ZSL Prediction Distribution")
                
                with gr.Row():
                    if os.path.exists("./data/zsl_top_predictions_analysis.png"):
                        gr.Image(value="./data/zsl_top_predictions_analysis.png", label="ZSL Top Predictions Analysis")
                
                with gr.Accordion("語義屬性說明", open=False):
                    gr.Markdown("""
                    **Zero-Shot Learning 語義屬性定義：**
                    - **high_performance**: 高性能表現 (基於歷史積分和排名)
                    - **experienced**: 經驗豐富 (基於參賽年數)
                    - **consistent**: 穩定性 (基於排名一致性評分)
                    - **competitive_car**: 有競爭力的賽車 (基於車隊表現)
                    - **home_advantage**: 主場優勢 (本國比賽)
                    - **momentum**: 近期表現趨勢
                    - **fastest_lap_ability**: 最快圈速能力
                    - **championship_contender**: 冠軍爭奪者 (綜合指標)
                    """)
        
        return demo

# ==================== 主程序 ====================
def main():
    """主執行流程 - 零樣本學習版本"""
    os.makedirs(Config.DATA_DIR, exist_ok=True)
    set_seeds()
    
    print("Starting Zero-Shot Learning F1 Winner Prediction...")
    
    # 數據載入和特徵工程
    winners, drivers, teams, fastest_laps = load_data()
    drivers_feat = create_features(drivers, teams, fastest_laps)
    modeling_df = build_dataset(winners, drivers_feat)
    
    # 數據分割
    modeling_df['race_id'] = modeling_df['year'].astype(str) + "_" + modeling_df['Grand Prix'].astype(str)
    unique_race_ids = modeling_df['race_id'].unique()
    
    if len(unique_race_ids) < 2:
        train_df = test_df = modeling_df.copy()
    else:
        train_ids, test_ids = train_test_split(unique_race_ids, test_size=0.2, random_state=Config.SEED)
        train_df = modeling_df[modeling_df['race_id'].isin(train_ids)].copy()
        test_df = modeling_df[modeling_df['race_id'].isin(test_ids)].copy()
    
    # 移除race_id並預處理
    for df in [train_df, test_df]:
        if 'race_id' in df.columns:
            df.drop(columns=['race_id'], inplace=True)
    
    train_df, test_df, encoders, scaler, cat_dims, winner_attrs, non_winner_attrs = preprocess_data(train_df, test_df)
    
    print("語義屬性已創建:")
    print(f"獲勝者屬性: {winner_attrs}")
    print(f"非獲勝者屬性: {non_winner_attrs}")
    
    # 創建數據加載器
    train_set = F1Dataset(train_df, winner_attrs, non_winner_attrs)
    test_set = F1Dataset(test_df, winner_attrs, non_winner_attrs)
    
    weights = [1. / (train_df[Config.TARGET_COL].value_counts().get(t, 1) + 1e-6) for t in train_df[Config.TARGET_COL]]
    sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights))
    
    train_loader = DataLoader(train_set, batch_size=Config.BATCH_SIZE, sampler=sampler)
    test_loader = DataLoader(test_set, batch_size=Config.BATCH_SIZE, shuffle=False)
    
    # 零樣本學習模型訓練
    cat_dims_ordered = [cat_dims[c] for c in Config.CAT_COLS]
    model = F1ZSLModel(cat_dims_ordered, len(Config.NUM_COLS), len(winner_attrs)).to(Config.DEVICE)
    
    if not os.path.exists(Config.MODEL_PATH):
        print("訓練零樣本學習模型...")
        train_model(model, train_loader, test_loader, winner_attrs, non_winner_attrs)
    else:
        print("載入已訓練的零樣本學習模型...")
    
    # 評估模型
    evaluate_model(model, test_loader, test_df, winner_attrs, non_winner_attrs)
    
    # 啟動零樣本學習界面
    predictor = F1ZSLPredictor()
    predictor.setup(model, encoders, scaler, drivers_feat, winners, winner_attrs, non_winner_attrs)
    demo = predictor.create_interface()
    
    print("啟動零樣本學習預測界面...")
    demo.launch(share=False)

if __name__ == '__main__':
    main()